# ESP32: huấn luyện và đánh giá theo recording

Notebook độc lập, được đóng gói từ pipeline đã kiểm thử trong project. Dùng 3 feature đúng firmware: distance_cm, distance_delta_3, distance_std_5. Train/validation/test tách recording và luôn đủ 3 lớp. Chọn model bằng validation; test chỉ đánh giá sau khi chọn. Không đổi seed để tìm điểm đẹp.

Chỉ có 3 recording APPROACHING và 3 EMERGENCY: một split chưa chứng minh khả năng tổng quát. Cần thu thêm phiên độc lập và khớp chu kỳ lấy mẫu trước khi nạp thiết bị.


In [ ]:
# Kaggle thường đã có TensorFlow. Bỏ comment nếu môi trường thiếu thư viện.
# %pip install tensorflow pandas numpy scikit-learn


In [ ]:
"""Causal features matching ESP32 and class-covered recording partitions."""
from __future__ import annotations

import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

LABELS = ["SAFE", "APPROACHING", "EMERGENCY"]
FEATURE_COLUMNS = ["distance_cm", "distance_delta_3", "distance_std_5"]


def load_dataset(path: str | Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    required = {"source_file", "elapsed_s", "distance_cm", "label"}
    if required - set(frame.columns):
        raise ValueError(f"Missing columns: {sorted(required - set(frame.columns))}")
    frame["label"] = frame["label"].astype(str).str.upper()
    for column in ["elapsed_s", "distance_cm"]:
        frame[column] = pd.to_numeric(frame[column], errors="raise")
    if frame[list(required)].isna().any().any() or not np.isfinite(frame[["elapsed_s", "distance_cm"]]).all().all():
        raise ValueError("Missing or non-finite training measurements")
    if set(frame["label"]) != set(LABELS):
        raise ValueError(f"Expected all labels {LABELS}")
    if not frame["distance_cm"].between(2, 450).all():
        raise ValueError("Sensor faults must be handled separately, not trained as SAFE")
    frame = frame.sort_values(["source_file", "elapsed_s"], kind="stable").reset_index(drop=True)
    if frame.groupby("source_file")["elapsed_s"].diff().dropna().le(0).any():
        raise ValueError("Timestamps must increase strictly within each recording")
    return frame


def make_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.sort_values(["source_file", "elapsed_s"], kind="stable").copy()
    grouped = out.groupby("source_file", sort=False)["distance_cm"]
    out["distance_delta_3"] = grouped.diff(3).fillna(0.0)
    out["distance_std_5"] = grouped.transform(lambda s: s.rolling(5, min_periods=1).std(ddof=1).fillna(0.0))
    return out


def recording_splits(frame: pd.DataFrame, seed: int = 42) -> dict[str, pd.DataFrame]:
    """Group whole single-label recordings; enforce all classes in every split."""
    grouped = frame.groupby("source_file")["label"]
    if grouped.nunique().ne(1).any():
        raise ValueError("Mixed-label sessions require a session-stratified split manifest")
    labels = grouped.first()
    rng = np.random.default_rng(seed)
    manifest = {name: [] for name in ["train", "validation", "test"]}
    for label in LABELS:
        sources = np.array(sorted(labels[labels == label].index))
        if len(sources) < 3:
            raise ValueError(f"{label}: need >=3 independent recordings for train/validation/test")
        sources = rng.permutation(sources)
        held = max(1, int(len(sources) * 0.2))
        manifest["test"].extend(sources[:held].tolist())
        manifest["validation"].extend(sources[held:2 * held].tolist())
        manifest["train"].extend(sources[2 * held:].tolist())
    parts = {name: frame[frame["source_file"].isin(sources)].copy() for name, sources in manifest.items()}
    for name, part in parts.items():
        if set(part["label"]) != set(LABELS):
            raise ValueError(f"{name} does not cover all classes")
    return parts


def prepare_data(path: str | Path, seed: int = 42) -> dict:
    raw = load_dataset(path)
    frames = {name: make_features(part) for name, part in recording_splits(raw, seed).items()}
    x = {name: part[FEATURE_COLUMNS].to_numpy(dtype=np.float32) for name, part in frames.items()}
    y = {name: part["label"].map({label: i for i, label in enumerate(LABELS)}).to_numpy(dtype=np.int32) for name, part in frames.items()}
    mean = x["train"].mean(axis=0)
    scale = np.where(x["train"].std(axis=0) < 1e-6, 1.0, x["train"].std(axis=0)).astype(np.float32)
    return {"raw": raw, "frames": frames, "x": x, "y": y, "mean": mean, "scale": scale,
            "normalized": {name: ((values - mean) / scale).astype(np.float32) for name, values in x.items()}}


def split_manifest(data: dict) -> dict:
    return {name: {"recordings": sorted(part["source_file"].unique().tolist()), "rows": len(part),
                   "class_counts": part["label"].value_counts().reindex(LABELS, fill_value=0).to_dict()}
            for name, part in data["frames"].items()}


def metrics(y_true: np.ndarray, predictions: np.ndarray) -> dict:
    report = classification_report(y_true, predictions, labels=[0, 1, 2], target_names=LABELS, output_dict=True, zero_division=0)
    return {"accuracy": float(accuracy_score(y_true, predictions)),
            "macro_f1": float(f1_score(y_true, predictions, labels=[0, 1, 2], average="macro", zero_division=0)),
            "emergency_recall": report["EMERGENCY"]["recall"], "approaching_recall": report["APPROACHING"]["recall"],
            "confusion_matrix": confusion_matrix(y_true, predictions, labels=[0, 1, 2]).tolist(), "per_class": report}


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


In [ ]:
"""Select on grouped validation, test once, export ESP32-compatible INT8.

Run: python -m ai_model.train_keras_tflite
Candidates are saved separately from the currently flashed firmware model.
"""
from __future__ import annotations
import argparse
from pathlib import Path
import numpy as np


def float_literal(value: float) -> str:
    text = format(float(value), ".9g")
    if "." not in text and "e" not in text.lower():
        text += ".0"
    return text + "f"


def write_c_array(blob: bytes, output: Path, mean: np.ndarray, scale: np.ndarray) -> None:
    output.mkdir(parents=True, exist_ok=True)
    count = len(FEATURE_COLUMNS)
    rows = [", ".join(f"0x{value:02x}" for value in blob[i:i + 16]) for i in range(0, len(blob), 16)]
    (output / "model_data.h").write_text(
        "#pragma once\nextern const unsigned char g_model_data[];\nextern const unsigned int g_model_data_len;\n"
        f"extern const float g_feature_mean[{count}];\nextern const float g_feature_scale[{count}];\n", encoding="utf-8")
    (output / "model_data.cc").write_text(
        '#include "model_data.h"\nalignas(16) const unsigned char g_model_data[] = {\n  '
        + ",\n  ".join(rows) + "\n};\nconst unsigned int g_model_data_len = sizeof(g_model_data);\n"
        + f"const float g_feature_mean[{count}] = {{{', '.join(map(float_literal, mean))}}};\n"
        + f"const float g_feature_scale[{count}] = {{{', '.join(map(float_literal, scale))}}};\n", encoding="utf-8")


def train_and_export(data: dict, output: Path, epochs: int = 150, seed: int = 42) -> dict:
    import tensorflow as tf
    from sklearn.utils.class_weight import compute_class_weight
    weights = compute_class_weight("balanced", classes=np.arange(3), y=data["y"]["train"])
    candidates, trained, histories = [], {}, {}
    for hidden in [(), (4,), (8,), (16, 8)]:
        tf.keras.utils.set_random_seed(seed)
        name = "linear" if not hidden else "mlp_" + "_".join(map(str, hidden))
        layers = [tf.keras.Input(shape=(len(FEATURE_COLUMNS),))]
        layers += [tf.keras.layers.Dense(size, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(1e-3)) for size in hidden]
        layers.append(tf.keras.layers.Dense(3, activation="softmax"))
        model = tf.keras.Sequential(layers, name=name)
        model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        history = model.fit(data["normalized"]["train"], data["y"]["train"],
                            validation_data=(data["normalized"]["validation"], data["y"]["validation"]),
                            epochs=epochs, batch_size=32, class_weight=dict(enumerate(map(float, weights))),
                            callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, min_delta=1e-4, restore_best_weights=True),
                                       tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=7, factor=0.5, min_lr=1e-5)], verbose=2)
        pred = model.predict(data["normalized"]["validation"], verbose=0).argmax(axis=1)
        candidates.append({"name": name, "parameters": model.count_params(), "validation": metrics(data["y"]["validation"], pred)})
        trained[name] = model
        histories[name] = {key: [float(value) for value in values] for key, values in history.history.items()}
    selected = max(candidates, key=lambda row: (row["validation"]["macro_f1"], row["validation"]["emergency_recall"], -row["parameters"]))
    model = trained[selected["name"]]
    train_pred = model.predict(data["normalized"]["train"], verbose=0).argmax(axis=1)
    float_pred = model.predict(data["normalized"]["test"], verbose=0).argmax(axis=1)
    rng = np.random.default_rng(seed)
    calibration = data["normalized"]["train"][rng.permutation(len(data["y"]["train"]))[:500]]
    def representative_data():
        for row in calibration:
            yield [row.reshape(1, -1).astype(np.float32)]
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_data
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    blob = converter.convert()
    interpreter = tf.lite.Interpreter(model_content=blob)
    interpreter.allocate_tensors()
    inp, out = interpreter.get_input_details()[0], interpreter.get_output_details()[0]
    assert inp["dtype"] == np.int8 and out["dtype"] == np.int8
    assert inp["shape"].tolist() == [1, len(FEATURE_COLUMNS)] and out["shape"].tolist() == [1, 3]
    input_scale, input_zero = inp["quantization"]
    scaled = data["normalized"]["test"] / input_scale
    rounded = np.sign(scaled) * np.floor(np.abs(scaled) + 0.5)  # C++ lround parity
    quantized = np.clip(rounded + input_zero, -128, 127).astype(np.int8)
    outputs = []
    for row in quantized:
        interpreter.set_tensor(inp["index"], row.reshape(1, -1))
        interpreter.invoke()
        outputs.append(interpreter.get_tensor(out["index"])[0].tolist())
    int8_pred = np.asarray(outputs).argmax(axis=1)
    report = {"selected": selected["name"], "candidates": candidates, "train": metrics(data["y"]["train"], train_pred),
              "test_float": metrics(data["y"]["test"], float_pred), "test_int8": metrics(data["y"]["test"], int8_pred),
              "float_int8_agreement": float(np.mean(float_pred == int8_pred)),
              "input_saturation_fraction": float(np.mean((rounded + input_zero < -128) | (rounded + input_zero > 127))),
              "split": split_manifest(data), "seed": seed}
    # Report the currently implemented firmware policy separately from raw AI.
    firmware_pred = np.where(data["x"]["test"][:, 0] <= 30, 2, int8_pred)
    report["test_firmware_policy"] = metrics(data["y"]["test"], firmware_pred)
    output.mkdir(parents=True, exist_ok=True)
    (output / "ultrasonic_safety_int8.tflite").write_bytes(blob)
    model.save(output / "selected_model.keras")
    write_c_array(blob, output, data["mean"], data["scale"])
    metadata = {"labels": LABELS, "features": FEATURE_COLUMNS, "window_size": 5, "std_ddof": 1,
                "normalization": {"mean": data["mean"].tolist(), "scale": data["scale"].tolist()},
                "input": {"shape": inp["shape"].tolist(), "scale": float(input_scale), "zero_point": int(input_zero), "dtype": "int8"},
                "output": {"shape": out["shape"].tolist(), "scale": float(out["quantization"][0]), "zero_point": int(out["quantization"][1]), "dtype": "int8"},
                "model_bytes": len(blob), "hard_distance_rule_cm": 30.0,
                "training_median_dt_s": float(data["raw"].groupby("source_file")["elapsed_s"].diff().median()),
                "operators": sorted({op["op_name"] for op in interpreter._get_ops_details() if op["op_name"] != "DELEGATE"})}
    write_json(output / "model_metadata.json", metadata)
    write_json(output / "metrics.json", report)
    write_json(output / "history.json", histories)
    write_json(output / "parity_vectors.json", {"raw_features": data["x"]["test"][:20].tolist(),
               "int8_input": quantized[:20].tolist(), "int8_output": outputs[:20], "expected_label_id": int8_pred[:20].tolist()})
    print(f"Selected on validation: {selected['name']}; test INT8 accuracy={report['test_int8']['accuracy']:.4f}")
    return report


In [ ]:
# Thay None bằng Path('đường/dẫn/dataset_3class.csv') nếu có nhiều dataset.
DATASET_PATH = None
candidates = []
for root in [Path.cwd()] + list(Path.cwd().parents)[:3]:
    candidates.extend([root / 'ai_model/data/single_sensor/processed_3class/dataset_3class.csv', root / 'data/single_sensor/processed_3class/dataset_3class.csv'])
for root in [Path('/kaggle/input'), Path('/content')]:
    if root.exists(): candidates.extend(sorted(root.rglob('dataset_3class.csv')))
available = sorted(set(path.resolve() for path in candidates if path.exists()))
if DATASET_PATH is None:
    if len(available) != 1: raise ValueError(f'Cần chỉ định đúng một dataset; tìm thấy: {available}')
    DATASET_PATH = available[0]
OUTPUT_DIR = Path('/kaggle/working/esp32_candidate') if Path('/kaggle/working').exists() else Path('esp32_candidate')
data = prepare_data(DATASET_PATH, seed=42)
print(json.dumps(split_manifest(data), indent=2, ensure_ascii=False))
print('Median dt:', data['raw'].groupby('source_file')['elapsed_s'].diff().median())


In [ ]:
report = train_and_export(data, OUTPUT_DIR, epochs=150, seed=42)
print(json.dumps({key: report[key] for key in ['selected','test_float','test_int8','test_firmware_policy','float_int8_agreement']}, indent=2))


In [ ]:
# Đường cong learning; test không được dùng cho early stopping.
import matplotlib.pyplot as plt
histories = json.loads((OUTPUT_DIR / 'history.json').read_text())
history = histories[report['selected']]
plt.plot(history['accuracy'], label='train accuracy')
plt.plot(history['val_accuracy'], label='validation accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.show()


In [ ]:
import shutil
print(shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR))


## Trước khi nạp ESP32

Kiểm tra model_metadata.json: input [1,3], output [1,3], feature order và mean/scale. Các file model_data.cc/h được copy cùng nhau. Đối chiếu parity_vectors.json trên thiết bị, đo thời gian Invoke và arena thực tế. Chỉ tin test_int8 là metric của AI; test_firmware_policy đo thêm luật firmware (<=30 cm luôn EMERGENCY; AI có thể nâng mức ở phía ngoài). Nếu hai metric khác nhau, kiểm tra nhãn ở vùng biên 30 cm.
